In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder

# --- 1. Data Loading and Cleaning ---

# Define file paths based on uploaded file names (corrected to actual Excel files)
FILE_PROFITABILITY_EXCEL = '2024q2referencetables.xlsx' # Sheet: Quarterly Rates
FILE_INVESTMENT_EXCEL = 'ukea2023q2table2corpssectorv2.xlsx' # Sheet: 3.1.7_B
FILE_BALANCE_EXCEL = 'nbsreferencetables2024.xlsx' # Sheets: Table 3, Table 5, Table 4

# --- Utility Function to Load and Clean ONS Quarterly Data (Excel) ---
def load_ons_quarterly_data_excel(filepath, sheet_name):
    # Load raw data, skipping header rows
    df_raw = pd.read_excel(filepath, sheet_name=sheet_name, header=3)

    # Rename the time column
    df_raw.rename(columns={df_raw.columns[0]: 'Time'}, inplace=True)

    # Drop the dataset identifier row
    df = df_raw.iloc[1:].copy()

    # Convert time format (e.g., '1997 Q1' to '1997-03-31')
    def quarter_to_date(q_str):
        if not isinstance(q_str, str):
            return np.nan

        parts = q_str.split(' ')
        if len(parts) == 2:
            year, q = parts
            q_map = {'Q1': '-03-31', 'Q2': '-06-30', 'Q3': '-09-30', 'Q4': '-12-31'}
            return f"{year}{q_map.get(q, '-01-01')}"
        else:
            return np.nan

    df['Date'] = df['Time'].apply(lambda x: pd.to_datetime(quarter_to_date(x), errors='coerce'))

    # Convert numerical columns, coercing errors to NaN
    cols_to_convert = df.columns[1:-1]
    for col in cols_to_convert:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    return df

# --- Utility Function to Load ONS Annual Balance Sheet Data (Excel) ---
def load_ons_annual_data_excel(filepath, sheet_name, sector_col_name):
    # Load raw data, skipping header rows
    df_raw = pd.read_excel(filepath, sheet_name=sheet_name, header=1)

    # --- Debugging: Print raw column names ---
    print(f"\nRaw columns in {filepath}, sheet {sheet_name}:\n{list(df_raw.columns)}")

    # Clean up column names by keeping only the relevant parts
    def clean_col_name(col):
        # Prioritize matching key financial columns directly based on observed raw column names
        if col.strip() == 'Non-financial assets': return 'Total Assets'
        if col.strip() == 'Financial assets and liabilities': return 'Financial liabilities Total'
        if col.strip() == 'Total': return 'Net worth' # Map 'Total' column, which likely represents Net worth

        # Specific cases for AN.11, AN.21, Inventories, etc.
        if 'AN.11' in col: return 'AN.11 Fixed assets Total'
        if 'AN.21' in col: return 'AN.21 Natural resources Total'
        if 'Inventories' in col: return 'Inventories'

        # Fallback to general cleaning if no specific match
        return col.split('£million at end year)')[0].strip() if '£million at end year)' in col else col.strip()

    df_raw.columns = [clean_col_name(col) for col in df_raw.columns]

    # The first column usually contains the year/time period
    df_raw.rename(columns={df_raw.columns[0]: 'Year'}, inplace=True)

    # Drop rows that are not numerical data (like header/note rows)
    df = df_raw[pd.to_numeric(df_raw['Year'], errors='coerce').notna()].copy()
    df['Year'] = df['Year'].astype(int)

    # Convert relevant financial columns to numeric
    expected_numeric_cols = ['Total Assets', 'Financial liabilities Total', 'Net worth']
    for col in expected_numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Calculate Total Liabilities and Total Assets from the cleaned columns
    if 'Financial liabilities Total' in df.columns:
        df['TotalLiabilities'] = df['Financial liabilities Total']
    else:
        # Handle cases where the column might still be missing, e.g., if the raw data differs
        print(f"Warning: 'Financial liabilities Total' column not found in {filepath} sheet {sheet_name}. Setting TotalLiabilities to NaN.")
        df['TotalLiabilities'] = np.nan

    if 'Total Assets' in df.columns:
        df['TotalAssets'] = df['Total Assets']
    else:
        print(f"Warning: 'Total Assets' column not found in {filepath} sheet {sheet_name}. Setting TotalAssets to NaN.")
        df['TotalAssets'] = np.nan

    # Add a Sector column for merging
    df['Sector'] = sector_col_name

    # Filter for necessary columns and return
    return df[['Year', 'Sector', 'TotalAssets', 'TotalLiabilities', 'Net worth']].dropna(subset=['TotalAssets', 'TotalLiabilities'])

# Load and process Profitability Data (Table 2)
df_profit = load_ons_quarterly_data_excel(FILE_PROFITABILITY_EXCEL, sheet_name='Table 2')

# Melt the dataframe to long format for easier plotting by sector
id_vars = ['Time', 'Date']
value_vars_net = [
    'UK Continental Shelf companies % [note 2] Net [note 4] [note 5]',
    'UK Non-Continental Shelf Private Non-Financial Corporations % - Net [note 4] [note 5]',
    'Manufacturing sector Private Non-Financial Corporations % - Net [note 4] [note 5]',
    'Service sector Private Non-Financial Corporations % - Net [note 4] [note 5]'
]
df_profit_long = df_profit.melt(id_vars=id_vars, value_vars=value_vars_net,
                                var_name='SectorFull', value_name='NetRateReturn')
df_profit_long['Sector'] = df_profit_long['SectorFull'].str.split('%').str[0].str.split(' - ').str[-1].str.replace(' sector', '').str.replace(' companies', '').str.strip()
df_profit_long = df_profit_long.dropna(subset=['NetRateReturn']).sort_values('Date')


# Load and process Balance Sheet Data (Annual)
df_pnfc = load_ons_annual_data_excel(FILE_BALANCE_EXCEL, sheet_name='Table 5', sector_col_name='Private Non-Financial')
df_nfc = load_ons_annual_data_excel(FILE_BALANCE_EXCEL, sheet_name='Table 3', sector_col_name='Non-Financial')

# Load and process Gross Fixed Capital Formation (Investment) Data (Quarterly)
# Table 3.1.7_B contains P.51g (Gross fixed capital formation) for Non-Financial Corps (S.11)
df_invest_raw = load_ons_quarterly_data_excel(FILE_INVESTMENT_EXCEL, sheet_name='3.1.7_B')
df_invest = df_invest_raw[['Date', 'Gross fixed capital formation']].copy()
df_invest.rename(columns={'Gross fixed capital formation': 'InvestmentAmount'}, inplace=True)
df_invest = df_invest.dropna(subset=['InvestmentAmount'])

# --- 2. Data Preparation for Individual Charts ---

# Chart 2 Prep: Asset Base vs Profitability (Latest Year)
# Use PNFC Balance Sheet (Annual) and Profitability (Annual Average)
latest_year = df_pnfc['Year'].max() # Set latest_year based on df_pnfc to ensure data availability
df_profit_annual = df_profit_long.groupby(['Sector', df_profit_long['Date'].dt.year])['NetRateReturn'].mean().reset_index()
df_profit_annual.rename(columns={'Date': 'Year'}, inplace=True)

# Merge latest annual profitability with PNFC balance sheet (using only PNFC as a proxy for the 'Sector' breakdown requested)
merged_c2 = pd.merge(df_pnfc[df_pnfc['Year'] == latest_year],
                     df_profit_annual[df_profit_annual['Year'] == latest_year],
                     on='Year',
                     how='inner',
                     suffixes=('_BS', '_PR'))

# Merge results for Chart 7 (Investment vs Next-Quarter Profitability Change)
# This requires a quarterly profitability time series by sector (df_profit_long) and quarterly Investment (df_invest)
# NOTE: Investment is for total Non-Financial Corps, so we'll apply it across sectors for correlation, which is an approximation.
df_econ_profit = df_profit_long[['Date', 'Sector', 'NetRateReturn']].copy()
df_econ_profit['NextQuarterReturn'] = df_econ_profit.groupby('Sector')['NetRateReturn'].shift(-1)

# Approximate Investment for each sector by joining the total NFC Investment by date
# We will drop the 'Sector' grouping momentarily for merging, then re-add it for context.
df_merge_c7 = pd.merge(df_econ_profit,
                       df_invest[['Date', 'InvestmentAmount']],
                       on='Date',
                       how='left')
df_merge_c7 = df_merge_c7.dropna(subset=['NextQuarterReturn', 'InvestmentAmount'])


# Chart 3 Prep: Correlation Heatmap
# Combine key variables into one quarterly/annual dataset for PNFC. This requires complex merging and calculation which is beyond simple join due to mix of data types and tables.
# For simplicity and to match the intent: We will create proxy variables for the NFC sector using the annual data.
df_corr_base = df_nfc.copy()
df_corr_base.rename(columns={'TotalAssets': 'TotalAssets_NFC',
                            'TotalLiabilities': 'TotalLiabilities_NFC'}, inplace=True)
df_corr_base = df_corr_base.set_index('Year')

# Calculate proxies for growth and ratio
df_corr_base['AssetGrowthRate'] = df_corr_base['TotalAssets_NFC'].pct_change() * 100
df_corr_base['LiabilityRatio'] = df_corr_base['TotalLiabilities_NFC'] / df_corr_base['TotalAssets_NFC'] * 100

# Add Investment (Annual sum of quarterly data)
df_invest_annual = df_invest.groupby(df_invest['Date'].dt.year)['InvestmentAmount'].sum().reset_index()
df_invest_annual.rename(columns={'Date': 'Year', 'InvestmentAmount': 'InvestmentAmount'}, inplace=True)
df_invest_annual = df_invest_annual.set_index('Year')

# Add Annual Average Net Rate of Return for All Non-Financial Corps (LRWW)
# This now needs to read from the Excel file as well, using the correct sheet
df_nfc_annual_profit = load_ons_quarterly_data_excel(FILE_PROFITABILITY_EXCEL, sheet_name='Table 2')

# Filter for the 'All Private Non-Financial Corporations % [note 1]: Net [note 4] [note 5]' column
df_nfc_annual_profit = df_nfc_annual_profit[['Date', 'All Private Non-Financial Corporations % [note 1]: Net [note 4] [note 5]']].copy()

# Extract year and profitability, then calculate annual average
df_nfc_annual_profit['Year'] = df_nfc_annual_profit['Date'].dt.year
df_nfc_annual_profit.rename(columns={
    'All Private Non-Financial Corporations % [note 1]: Net [note 4] [note 5]': 'NetRateReturn'
}, inplace=True)

# Group by the extracted year and take the mean to get annual average profitability
df_nfc_annual_profit = df_nfc_annual_profit.groupby('Year')['NetRateReturn'].mean().reset_index()
df_nfc_annual_profit = df_nfc_annual_profit.set_index('Year').dropna()

# Final correlation merge
df_corr = pd.concat([df_corr_base, df_invest_annual, df_nfc_annual_profit], axis=1).dropna()
vars_for_corr = ['AssetGrowthRate', 'LiabilityRatio', 'InvestmentAmount', 'NetRateReturn']


# Chart 4 Prep: Sector Contribution to Total Assets Over Time
# Use the PNFC Balance Sheet for annual asset data for the Private Non-Financial sector
# The ONS profitability data tables don't contain absolute asset values by sub-sector (Manuf, Service, UKCS) over time.
# To generate a Stacked Area Chart, we will use the Total Assets (non-financial assets, AN.1) from the three main PNFC categories:
# NFC Total (Table 3): 'Total non-financial assets'
# PC Total (Table 4): 'Total non-financial assets'
# This requires loading Table 4 (Public Non-Financial Corps)
df_pc = load_ons_annual_data_excel(FILE_BALANCE_EXCEL, sheet_name='Table 4', sector_col_name='Public Non-Financial')
df_nfc_assets = df_nfc[['Year', 'TotalAssets']].rename(columns={'TotalAssets': 'Non-Financial Corp Assets'})
df_pc_assets = df_pc[['Year', 'TotalAssets']].rename(columns={'TotalAssets': 'Public Corp Assets'})

df_assets_total = pd.merge(df_nfc_assets, df_pc_assets, on='Year', how='inner')
df_assets_total['Private Non-Financial Corp Assets'] = df_pnfc['TotalAssets'] # Assuming PNFC assets are a component of NFC (S.11) assets, which is true, but using the full NFC assets and PC assets for a simple 2-way stack to avoid double counting

# For a better stacked area: use Private NFC and Public Corp assets as two main components
df_pivot = df_assets_total[['Year', 'Private Non-Financial Corp Assets', 'Public Corp Assets']].set_index('Year').tail(15) # Last 15 years

# Chart 6 Prep: Liabilities vs Net Worth Trend (2010–2023) for Non-Financial Corporations
df_lw_nfc = df_nfc.copy()
df_lw_nfc = df_lw_nfc[(df_lw_nfc['Year'] >= 2010)]

# --- 3. Chart Generation ---
plt.style.use('seaborn-v0_8-whitegrid')

# --- Chart 1: Profitability Trend by Sector (2010-2024) ---
plt.figure(figsize=(12, 6))
# Filter data from 2010 onwards
df_c1 = df_profit_long[df_profit_long['Date'].dt.year >= 2010]
sns.lineplot(data=df_c1, x='Date', y='NetRateReturn', hue='Sector')
plt.title('Net Rate of Return by Sector (Quarterly, 2010-2024)', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Net Rate of Return (%)', fontsize=12)
plt.legend(title='Sector', loc='upper left')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# --- Chart 2: Asset Base vs Profitability (Latest Year) ---
plt.figure(figsize=(10, 8))
# Use Total Assets (£m) on the x-axis and Net Rate of Return (%) on the y-axis
plt.scatter(merged_c2['TotalAssets'], merged_c2['NetRateReturn'],
            s=merged_c2['TotalLiabilities']/merged_c2['TotalAssets'] * 1000,
            alpha=0.6, edgecolors='w', linewidth=0.5)

for i, sector in enumerate(merged_c2['Sector_PR']):
    plt.annotate(sector, (merged_c2['TotalAssets'].iat[i], merged_c2['NetRateReturn'].iat[i]),
                 textcoords="offset points", xytext=(5,-5), ha='left')

plt.xscale('log')
plt.title(f'Assets vs Rate of Return by PNFC Sector (Latest Annual Average – {latest_year})', fontsize=14)
plt.xlabel('Total Non-Financial Assets (£ million, log scale)', fontsize=12)
plt.ylabel('Net Rate of Return (Annual Avg, %)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# --- Chart 3: Correlation Heatmap: Asset Growth, Investment, Liability Ratio vs Profitability ---
corr = df_corr[vars_for_corr].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', cbar_kws={'label': 'Correlation Coefficient'})
plt.title('Correlation Matrix – Key NFC Variables vs Profitability (Annual)', fontsize=14)
plt.tight_layout()
plt.show()

# --- Chart 4: Sector Contribution to Total Assets Over Time (2010–2023) ---
df_pivot.plot(kind='area', stacked=True, figsize=(12, 6), colormap='viridis')
plt.title('Non-Financial Corporations Total Non-Financial Assets by Sector (2010-2023)', fontsize=14)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Assets (£ million)', fontsize=12)
plt.legend(loc='upper left', title='Sector')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# --- Chart 5: Distribution of Rate of Return by Sector (Latest Year) ---
plt.figure(figsize=(10, 6))
df_c5 = df_profit_long[df_profit_long['Date'].dt.year == latest_year]
sns.boxplot(data=df_c5, x='Sector', y='NetRateReturn', palette='Set2')
plt.title(f'Distribution of Net Rate of Return by PNFC Sector (Quarterly – {latest_year})', fontsize=14)
plt.xlabel('PNFC Sub-Sector', fontsize=12)
plt.ylabel('Net Rate of Return (%)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# --- Chart 6: Liabilities vs Net Worth Trend (2010–2023) for Non-Financial Corporations ---
plt.figure(figsize=(12, 6))
plt.plot(df_lw_nfc['Year'], df_lw_nfc['TotalLiabilities'], label='Total Financial Liabilities', marker='o')
plt.plot(df_lw_nfc['Year'], df_lw_nfc['Net worth'], label='Net Worth', marker='s')
plt.title('Financial Liabilities vs Net Worth – Non-Financial Corporations (2010-2023)', fontsize=14)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Value (£ million)', fontsize=12)
plt.legend(loc='upper left')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# --- Chart 7: Investment (GFCF) vs Next-Quarter Profitability Change (Quarterly) ---
plt.figure(figsize=(10, 8))
sns.scatterplot(data=df_merge_c7, x='InvestmentAmount', y='NextQuarterReturn', hue='Sector', alpha=0.7)
plt.xscale('log')
plt.title('GFCF (Proxy for Investment) vs Next Quarter Rate of Return (Quarterly)', fontsize=14)
plt.xlabel('Non-Financial Corp GFCF (£ million, log scale)', fontsize=12)
plt.ylabel('Next Quarter Net Rate of Return (%)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# --- Chart 8: Feature Importance (Model) – What Drives Profitability? (Synthetic Example) ---
# This is a synthetic example as requested, using derived annual data.

# Prep data for RF model
df_model = df_corr.dropna().copy()

# Add a categorical feature proxy for Sector using PNFC data
le = LabelEncoder()
# Create a synthetic dataset that merges PNFC sector profitability with the annual NFC metrics
df_pnfc_sector_annual = df_profit_annual[df_profit_annual['Sector'] != 'All Private Non-Financial Corporations'].copy()
df_pnfc_sector_annual = df_pnfc_sector_annual.pivot(index='Year', columns='Sector', values='NetRateReturn')
df_pnfc_sector_annual = df_pnfc_sector_annual.apply(lambda x: x.fillna(x.mean()), axis=0) # Simple imputation
df_model = df_model.merge(df_pnfc_sector_annual, on='Year', how='inner')
df_model.rename(columns={'Manufacturing Private Non-Financial Corporations': 'NetReturn_Manufacturing',
                         'Service Private Non-Financial Corporations': 'NetReturn_Service',
                         'UK Continental Shelf': 'NetReturn_UKCS',
                         'UK Non-Continental Shelf Private Non-Financial Corporations': 'NetReturn_NonUKCS'}, inplace=True)

# Define target and features
y = df_model['NetRateReturn'] # Overall NFC Profitability
features = ['AssetGrowthRate', 'LiabilityRatio', 'InvestmentAmount', 'NetReturn_Manufacturing', 'NetReturn_Service']
X = df_model[features].dropna()
y = y.loc[X.index] # Align y to X index after dropping NaNs

# Re-run model only if enough data points exist (at least 2, preferably > 10)
if len(X) > 10:
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X, y)

    importances = pd.Series(model.feature_importances_, index=features)

    plt.figure(figsize=(8, 6))
    importances.sort_values().plot(kind='barh', figsize=(8, 6), color='teal')
    plt.title('Feature Importances – Predicting Overall PNFC Net Rate of Return (Synthetic)', fontsize=14)
    plt.xlabel('Relative Importance (Random Forest)', fontsize=12)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("\n[Image of Feature Importance Chart Placeholder]")
    print(f"Skipping Chart 8 due to insufficient merged data points ({len(X)} rows) after cleaning and merging annual time series data.")

FileNotFoundError: [Errno 2] No such file or directory: '2024q2referencetables.xlsx'